In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector


load_dotenv(override=True)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
    embeddings=embedding_model,
    collection_name="university_rules",
    connection=connection_url,
    use_jsonb=True,
)

print("PostgreSQL + pgvector 연결 및 초기화 완료")

In [ ]:
# LangChain의 기본 문서 단위인 Document 클래스를 불러옵니다.
from langchain_core.documents import Document

# 설계한 메타데이터 스키마를 적용한 규정 문서 리스트를 생성합니다.
documents_to_insert = [
    # 1번 문서: 2026학년도 도서관 대출 규정
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 도서 대출이 가능합니다.",
        metadata={
            "source": "library_guide_2026.pdf",
            "category": "library",
            "year": 2026
        }
    ),
    # 2번 문서: 2027학년도 개정 도서관 대출 규정
    Document(
        page_content="[개정 도서관 규정] 학부생은 최대 7권까지 21일간 도서 대출이 가능합니다.",
        metadata={
            "source": "library_guide_2027.pdf",
            "category": "library",
            "year": 2027
        }
    ),
    # 3번 문서: 2027학년도 장학금 규정
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상인 학생은 성적 우수 장학금 대상자입니다.",
        metadata={
            "source": "scholarship_2027.pdf",
            "category": "scholarship",
            "year": 2027
        }
    )
]

# 각 문서에 일대일로 매핑될 고유 식별자(ID)를 지정합니다.
document_ids = ["doc_lib_2026", "doc_lib_2027", "doc_sch_2027"]

# Document 객체와 고유 ID를 데이터베이스에 적재(Upsert)합니다.
saved_ids = vector_store.add_documents(
    documents=documents_to_insert,
    ids=document_ids
)

# 저장 완료 결과를 화면에 출력합니다.
print(f"DB에 적재된 문서 총 개수: {len(saved_ids)}개")
print(f"저장된 문서 고유 ID 목록: {saved_ids}")

In [ ]:
metadata_filter = {
    "category": "library",
    "year": 2027
}

search_results = vector_store.similarity_search(
    query="최대 대출 일수를 알려주세요.",    
    k=1,  # 가장 유사한 1개의 결과를 반환
    filter=metadata_filter  # 메타데이터 필터 적용
)

search_docs = search_results[0]
print(f"검색 본문  : {search_docs.page_content}")
print(f"메타데이터 : {search_docs.metadata}")